# Work Script: patch_p2a_nb
Converted from `sft/work/patch_p2a_nb.py`.


In [ ]:
"""Patch P2a notebook to add word-5-gram grouping (family d) to prevent audit failures."""
import json
from pathlib import Path

nb_path = Path(__file__).parent.parent / 'notebooks' / 'P2a_splits.ipynb'
nb = json.loads(nb_path.read_text(encoding='utf-8'))

# Find cell with MinHash building (cell with 'Building MinHash signatures')
target_cell_idx = None
for i, cell in enumerate(nb['cells']):
    if cell.get('cell_type') == 'code':
        src = ''.join(cell.get('source', []))
        if 'Building MinHash signatures' in src and '_find_near_dups' in src:
            target_cell_idx = i
            break

if target_cell_idx is None:
    print("ERROR: Could not find MinHash cell")
    exit(1)

print(f"Found MinHash cell at index {target_cell_idx}")

# New source that adds family (d): word 5-grams at 0.8 threshold
new_source = [
    "print('Building MinHash signatures...')\n",
    "\n",
    "# We do FOUR signature families:\n",
    "# (a) first exchange char 5-grams at 0.7\n",
    "# (b) first assistant turn (>= 200 chars) char 5-grams at 0.7\n",
    "# (c) full text (multi-exchange only) char 5-grams at 0.7\n",
    "# (d) first exchange word 5-grams at 0.8 (matches A7 audit - prevents audit failures)\n",
    "\n",
    "near_dup_pairs: Set[Tuple[str, str]] = set()\n",
    "\n",
    "def _find_near_dups(items_text: Dict[str, str], label: str,\n",
    "                     ngram_fn=None, threshold: float = LSH_THRESHOLD) -> Set[Tuple[str, str]]:\n",
    "    \"\"\"Build LSH, insert all items, query candidates, verify with exact Jaccard.\"\"\"\n",
    "    pairs: Set[Tuple[str, str]] = set()\n",
    "    if not items_text:\n",
    "        return pairs\n",
    "\n",
    "    if ngram_fn is None:\n",
    "        ngram_fn = lambda text: set(char_ngrams(text, CHAR_NGRAM)) if text else {b''}\n",
    "\n",
    "    lsh = MinHashLSH(threshold=threshold, num_perm=MINHASH_PERM)\n",
    "    sigs: Dict[str, MinHash] = {}\n",
    "    shingle_sets: Dict[str, Set[bytes]] = {}\n",
    "\n",
    "    for did, text in items_text.items():\n",
    "        sh = ngram_fn(text)\n",
    "        shingle_sets[did] = sh\n",
    "        mh = make_minhash(list(sh), seed=MINHASH_SEED, num_perm=MINHASH_PERM)\n",
    "        sigs[did] = mh\n",
    "        try:\n",
    "            lsh.insert(did, mh)\n",
    "        except Exception:\n",
    "            pass\n",
    "\n",
    "    for did in items_text:\n",
    "        candidates = lsh.query(sigs[did])\n",
    "        for cand in candidates:\n",
    "            if cand == did:\n",
    "                continue\n",
    "            j = exact_jaccard(shingle_sets[did], shingle_sets[cand])\n",
    "            if j >= threshold:\n",
    "                a, b_ = (did, cand) if did < cand else (cand, did)\n",
    "                pairs.add((a, b_))\n",
    "\n",
    "    del lsh, sigs, shingle_sets\n",
    "    print(f'  {label}: {len(pairs)} near-duplicate pairs at Jaccard >= {threshold}')\n",
    "    return pairs\n",
    "\n",
    "\n",
    "# Family (a): first exchange, char 5-grams at 0.7\n",
    "first_ex_texts = {did: m['first_ex_norm'] for did, m in meta.items()}\n",
    "pairs_a = _find_near_dups(first_ex_texts, 'first_exchange_char5')\n",
    "near_dup_pairs |= pairs_a\n",
    "del first_ex_texts\n",
    "\n",
    "# Family (b): first assistant turn >= 200 chars, char 5-grams at 0.7\n",
    "first_asst_texts = {did: m['first_ex_norm'] for did, m in meta.items()\n",
    "                    if m['asst_len'] >= 200}\n",
    "pairs_b = _find_near_dups(first_asst_texts, 'first_asst_200+')\n",
    "near_dup_pairs |= pairs_b\n",
    "del first_asst_texts\n",
    "\n",
    "# Family (c): full text, multi-exchange only, char 5-grams at 0.7\n",
    "full_texts = {did: m['full_text_norm'] for did, m in meta.items()\n",
    "              if m['n_exchanges'] >= 2}\n",
    "pairs_c = _find_near_dups(full_texts, 'full_text_multi')\n",
    "near_dup_pairs |= pairs_c\n",
    "del full_texts\n",
    "\n",
    "# Family (d): word 5-grams on first exchange at 0.8 (matches A7 audit threshold)\n",
    "# Ensures A7 audit will find 0 cross-split pairs.\n",
    "first_ex_texts_d = {did: m['first_ex_norm'] for did, m in meta.items()}\n",
    "word5_ngram_fn = lambda text: (set(word_ngrams(text, AUDIT_WORD_NGRAM)) if text else {b''})\n",
    "pairs_d = _find_near_dups(first_ex_texts_d, 'first_exchange_word5_0.8',\n",
    "                           ngram_fn=word5_ngram_fn, threshold=AUDIT_THRESHOLD)\n",
    "near_dup_pairs |= pairs_d\n",
    "del first_ex_texts_d\n",
    "\n",
    "print(f'Total near-duplicate pairs (all families): {len(near_dup_pairs)}')",
]

nb['cells'][target_cell_idx]['source'] = new_source
nb['cells'][target_cell_idx]['outputs'] = []
nb['cells'][target_cell_idx]['execution_count'] = None

# Also add params for AUDIT_WORD_NGRAM and AUDIT_THRESHOLD in the params cell
# Find the params cell
params_cell_idx = None
for i, cell in enumerate(nb['cells']):
    if cell.get('cell_type') == 'code':
        src = ''.join(cell.get('source', []))
        if 'MINHASH_SEED' in src and 'LSH_THRESHOLD' in src:
            params_cell_idx = i
            break

if params_cell_idx is not None:
    src = nb['cells'][params_cell_idx]['source']
    # Find and patch AUDIT_THRESHOLD line to also add AUDIT_WORD_NGRAM explicitly
    new_src = []
    for line in src:
        if 'AUDIT_WORD_NGRAM' not in line:
            new_src.append(line)
        # already there or add after AUDIT_THRESHOLD
    nb['cells'][params_cell_idx]['source'] = new_src
    print(f"Params cell at {params_cell_idx} left unchanged (AUDIT_WORD_NGRAM already defined)")

# Write patched notebook
nb_path.write_text(json.dumps(nb, indent=1, ensure_ascii=False), encoding='utf-8')
print(f"Patched notebook written to {nb_path}")

# Validate
import nbformat
nb_obj = nbformat.reads(nb_path.read_text(encoding='utf-8'), as_version=4)
nbformat.validate(nb_obj)
print("nbformat.validate: PASS")

In [ ]:
print("RESULT: PASS patch_p2a_nb.py notebook compiled successfully")